# Archive v1 (Colab) · test yolov8n

Prédictions du modèle YOLOv8n à 60 classes sur des images `val_images` de xView.

*Non utilisé par le modèle final. Conservé pour retracer l'itération ; images retirées des sorties.*

In [ ]:
# --- Configuration des chemins ---
# Exécution d'origine sur Google Colab :
#   DATA_LAKE : dossier « data_lake » sur Google Drive (sous-dossiers xView/ et DOTA/)
#   WORK_DIR  : disque local de la machine Colab (/content)
DATA_LAKE = "/content/drive/MyDrive/data_lake"
WORK_DIR = "/content"

In [4]:
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

Mounted at /content/drive


In [5]:
drive_runs = f"{DATA_LAKE}/xView/runs/xview_yolov8n"
local_runs = f"{WORK_DIR}/xView_local/runs/xview_yolov8n"

os.makedirs(os.path.dirname(local_runs), exist_ok=True)
if not os.path.exists(local_runs):
    shutil.copytree(drive_runs, local_runs)
    print("Modèle copié en local.")
else:
    print("Déjà présent en local.")

model_path = os.path.join(local_runs, "weights", "best.pt")
print(f"Modèle présent : {os.path.exists(model_path)}")

Déjà présent en local.
Modèle présent : True


In [8]:
drive_val_images = f"{DATA_LAKE}/xView/val_images"
local_val_images = f"{WORK_DIR}/xView_local/val_images"

if not os.path.exists(local_val_images):
    print("Copie des images de validation...")
    shutil.copytree(drive_val_images, local_val_images)
    print("Terminé.")
else:
    print("Déjà présent en local.")

n_val = len([f for f in os.listdir(local_val_images) if f.endswith('.tif') and not f.startswith('._')])
print(f"Images de validation disponibles : {n_val}")

Copie des images de validation...
Terminé.
Images de validation disponibles : 281


In [9]:
n_val = len([f for f in os.listdir(local_val_images) if f.endswith('.tif')])
print(f"Images de validation en local : {n_val}")

Images de validation en local : 281


In [10]:
!pip install ultralytics -q

In [11]:
from ultralytics import YOLO
model = YOLO(model_path)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [20]:
sample_images = [f for f in os.listdir(local_val_images) if f.endswith('.tif') and not f.startswith('._')][:10]

for img_name in sample_images:
    img_path = os.path.join(local_val_images, img_name)
    results = model.predict(img_path, save=True, conf=0.20, project=f"{WORK_DIR}/xView_local/runs", name="inference_demo")
    print(f"{img_name} : {len(results[0].boxes)} objets détectés")


image 1/1 /content/xView_local/val_images/2428.tif: 544x640 11 Small Cars, 1 Truck, 36 Buildings, 2 Pylons, 169.3ms
Speed: 6.7ms preprocess, 169.3ms inference, 2.1ms postprocess per image at shape (1, 3, 544, 640)
Results saved to /content/xView_local/runs/inference_demo-2
2428.tif : 50 objets détectés

image 1/1 /content/xView_local/val_images/963.tif: 608x640 2 Small Cars, 57 Buildings, 179.3ms
Speed: 6.4ms preprocess, 179.3ms inference, 1.3ms postprocess per image at shape (1, 3, 608, 640)
Results saved to /content/xView_local/runs/inference_demo-2
963.tif : 59 objets détectés

image 1/1 /content/xView_local/val_images/690.tif: 608x640 (no detections), 175.9ms
Speed: 6.1ms preprocess, 175.9ms inference, 0.7ms postprocess per image at shape (1, 3, 608, 640)
Results saved to /content/xView_local/runs/inference_demo-2
690.tif : 0 objets détectés

image 1/1 /content/xView_local/val_images/2570.tif: 448x640 4 Small Cars, 2 Motorboats, 3 Sailboats, 1 Fishing Vessel, 289 Buildings, 1 Stor

In [25]:
import matplotlib.pyplot as plt
from PIL import Image

demo_dir = f"{WORK_DIR}/xView_local/runs/inference_demo-2"
demo_files = [f for f in os.listdir(demo_dir) if f.endswith(('.jpg', '.png'))]

if demo_files:
    img = Image.open(os.path.join(demo_dir, demo_files[0]))
    plt.figure(figsize=(14, 14))
    plt.imshow(img)
    plt.axis('off')
    plt.title(demo_files[0])
    plt.show()

In [27]:
drive_demo = f"{DATA_LAKE}/xView/inference_demo"
if not os.path.exists(drive_demo):
    shutil.copytree(demo_dir, drive_demo)
    print("Résultats de démo sauvegardés sur Drive.")

Résultats de démo sauvegardés sur Drive.


In [17]:
results = model.predict(f"{WORK_DIR}/xView_local/val_images/636.tif", save=True, conf=0.10, project=f"{WORK_DIR}/xView_local/runs", name="debug_low_conf")


image 1/1 /content/xView_local/val_images/636.tif: 608x640 1 Building, 5 Pylons, 187.1ms
Speed: 6.4ms preprocess, 187.1ms inference, 1.1ms postprocess per image at shape (1, 3, 608, 640)
Results saved to /content/xView_local/runs/debug_low_conf


In [19]:
from PIL import Image
import matplotlib.pyplot as plt

f1_curve = Image.open(f"{WORK_DIR}/xView_local/runs/xview_yolov8n/BoxF1_curve.png")
pr_curve = Image.open(f"{WORK_DIR}/xView_local/runs/xview_yolov8n/BoxPR_curve.png")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(f1_curve)
axes[0].axis('off')
axes[0].set_title("F1 vs Confidence")
axes[1].imshow(pr_curve)
axes[1].axis('off')
axes[1].set_title("Precision-Recall")
plt.show()